In [1]:
!pip install fastapi uvicorn httpx

  Using cached httpx-0.28.1-py3-none-any.whl (73 kB)


In [9]:
import sys

print(sys.version)
print(sys.executable)

3.8.5 (default, Sep  3 2020, 21:29:08) [MSC v.1916 64 bit (AMD64)]
C:\Users\vaio hd\anaconda3\python.exe


In [10]:
%%writefile main.py

from typing import List

from fastapi import FastAPI, HTTPException
from pydantic import BaseModel


app = FastAPI(
    title="Products API",
    description="API مدیریت محصولات",
    version="1.0.0"
)


class Product(BaseModel):
    id: int
    name: str
    price: float


class ProductCreate(BaseModel):
    name: str
    price: float


products: List[Product] = [
    Product(
        id=1,
        name="Laptop",
        price=1200
    ),
    Product(
        id=2,
        name="Mouse",
        price=25
    ),
    Product(
        id=3,
        name="Keyboard",
        price=50
    )
]


@app.get("/")
def home():
    return {
        "message": "Products API is running"
    }


@app.get("/products", response_model=List[Product])
def get_products():
    return products


@app.get("/products/{product_id}", response_model=Product)
def get_product(product_id: int):
    for product in products:
        if product.id == product_id:
            return product

    raise HTTPException(
        status_code=404,
        detail="Product not found"
    )


@app.post("/products", response_model=Product, status_code=201)
def create_product(product_data: ProductCreate):
    if len(products) == 0:
        new_id = 1
    else:
        new_id = max(product.id for product in products) + 1

    new_product = Product(
        id=new_id,
        name=product_data.name,
        price=product_data.price
    )

    products.append(new_product)

    return new_product


@app.put("/products/{product_id}", response_model=Product)
def update_product(
    product_id: int,
    product_data: ProductCreate
):
    for index, product in enumerate(products):
        if product.id == product_id:
            updated_product = Product(
                id=product_id,
                name=product_data.name,
                price=product_data.price
            )

            products[index] = updated_product

            return updated_product

    raise HTTPException(
        status_code=404,
        detail="Product not found"
    )


@app.delete("/products/{product_id}")
def delete_product(product_id: int):
    for index, product in enumerate(products):
        if product.id == product_id:
            deleted_product = products.pop(index)

            return {
                "message": "Product deleted successfully",
                "product": deleted_product
            }

    raise HTTPException(
        status_code=404,
        detail="Product not found"
    )

Writing main.py


In [11]:
import os

print(os.getcwd())
print(os.listdir())

C:\Users\vaio hd
['.android', '.AndroidStudio3.4', '.conda', '.condarc', '.gradle', '.idlerc', '.ipynb_checkpoints', '.ipython', '.jupyter', '.keras', '.kivy', '.matplotlib', '.nbi', '.pylint.d', '.spyder-py3', '.stmcufinder', '.vscode', '1tls_2x2.csv', '1tls_3x3.csv', '1tls_4x4.csv', '2tls_3x3x2.csv', '3D Objects', '4tls_3x3x2x2.csv', 'Adaptive_TSC.ipynb', 'AI_traci.ipynb', 'AI_Traffic.ipynb', 'anaconda3', 'AndroidStudioProjects', 'AppData', 'Application Data', 'best_weights.pt', 'BUS.ipynb', 'Bus1.ipynb', 'Bus2.ipynb', 'Contacts', 'Cookies', 'cross.sumocfg', 'cross1.sumocfg', 'cross3ltl.sumocfg', 'Desktop', 'directory to extract', 'Documents', 'Downloads', 'DQN.h5', 'DQNet.h5', 'DQN_ITS.ipynb', 'drlsc-rp.ipynb', 'EMG1.ipynb', 'EMG2.ipynb', 'Favorites', 'fourth.net.xml', 'four_junction_grid.sumocfg', 'IMP.zip', 'input_routes.rou.xml', 'IntelGraphicsProfiles', 'intersection.net.xml', 'intersection.rou.xml', 'intersection.sumocfg', 'INT_sumo.ipynb', 'length.ipynb', 'Links', 'Local Setti

# اجرای سرور FastAPI

In [12]:
import subprocess
import sys
import time

server = subprocess.Popen(
    [
        sys.executable,
        "-m",
        "uvicorn",
        "main:app",
        "--port",
        "8000"
    ]
)

time.sleep(2)

print("سرور اجرا شد:")
print("http://127.0.0.1:8000")
print("http://127.0.0.1:8000/docs")

سرور اجرا شد:
http://127.0.0.1:8000
http://127.0.0.1:8000/docs


In [13]:
import requests

response = requests.get("http://127.0.0.1:8000")

print(response.status_code)
print(response.json())

200
{'message': 'Products API is running'}


# دریافت همه محصولات

In [14]:
response = requests.get(
    "http://127.0.0.1:8000/products"
)

print(response.status_code)
print(response.json())

200
[{'id': 1, 'name': 'Laptop', 'price': 1200.0}, {'id': 2, 'name': 'Mouse', 'price': 25.0}, {'id': 3, 'name': 'Keyboard', 'price': 50.0}]


# ایجاد محصول جدید

In [15]:
new_product = {
    "name": "Monitor",
    "price": 300
}

response = requests.post(
    "http://127.0.0.1:8000/products",
    json=new_product
)

print(response.status_code)
print(response.json())

201
{'id': 4, 'name': 'Monitor', 'price': 300.0}


# به روز رسانی

In [16]:
updated_product = {
    "name": "Wireless Mouse",
    "price": 35
}

response = requests.put(
    "http://127.0.0.1:8000/products/2",
    json=updated_product
)

print(response.status_code)
print(response.json())

200
{'id': 2, 'name': 'Wireless Mouse', 'price': 35.0}


# حذف محصول

In [17]:
response = requests.delete(
    "http://127.0.0.1:8000/products/3"
)

print(response.status_code)
print(response.json())

200
{'message': 'Product deleted successfully', 'product': {'id': 3, 'name': 'Keyboard', 'price': 50.0}}


# خاموش کردن سرور 

In [19]:
server.terminate()
server.wait()

print("server shut down")

server shut down
